# AG Sampler — Neal's Funnel Experiment

Validates the core hypothesis: does online score-matched flow training
improve MCLMC sampling on a hard posterior?

**Open in Colab:** [click here](https://colab.research.google.com/github/amospagin/mcmc-estimation/blob/main/notebooks/funnel_experiment.ipynb)

In [ ]:
# Install
!rm -rf /content/ag
!git clone https://github.com/amospagin/mcmc-estimation.git /content/ag
!ls /content/ag/agsampler/__init__.py
!pip install -q jax jaxlib numpy arviz matplotlib optax equinox

import sys
sys.path.insert(0, '/content/ag')

# Verify
import agsampler
print(f"AG Sampler version: {agsampler.__version__}")

In [ ]:
import jax
print(f"JAX devices: {jax.devices()}")

In [ ]:
from experiments.neal_funnel import run_experiment

results = run_experiment(
    dim=10,
    num_chains=4,
    num_samples=2000,
    warmup_steps=1000,
    seed=42,
)

In [ ]:
# Plot flow training loss (did the flow learn?)
import matplotlib.pyplot as plt

if 'hybrid_flow' in results and 'flow_losses' in results['hybrid_flow']:
    losses = results['hybrid_flow']['flow_losses']
    if losses:
        plt.figure(figsize=(8, 3))
        plt.plot(losses)
        plt.xlabel('Training step')
        plt.ylabel('Score matching loss')
        plt.title('Flow training loss (hybrid NUTS warmup -> score matching)')
        plt.yscale('log')
        plt.tight_layout()
        plt.show()
    else:
        print('No flow losses recorded')
else:
    print('Hybrid flow experiment did not run')

In [ ]:
# Quick visual: v marginal (should be N(0, 9))
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)

for ax, (name, key) in zip(axes, [
    ('MCLMC', 'mclmc_baseline'),
    ('Hybrid (ours)', 'hybrid_flow'),
    ('NUTS', 'nuts_baseline'),
]):
    if key not in results or 'error' in results[key]:
        ax.set_title(f'{name} (failed)')
        continue
    r = results[key]
    ax.set_title(f"{name}\nR-hat={r['rhat_max']:.3f}, v_var={r['v_var']:.2f}")
    ax.axhline(9.0, color='red', linestyle='--', alpha=0.5, label='true var')
    ax.bar(['v_var'], [r['v_var']])
    ax.set_ylim(0, 15)

plt.suptitle("Neal's Funnel: variance of v (true = 9.0)")
plt.tight_layout()
plt.show()